# Bangla SmolLM 135m Training

Training rnnandi/bangla-smollm-135m with QLoRA on Banglish emotion-tagged data.


In [ ]:
%%capture
!pip install -q transformers peft bitsandbytes datasets accelerate sentencepiece pyyaml huggingface_hub

## Imports & Setup


In [ ]:
import os
import json
import random

import torch
import numpy as np
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

OUTPUT_DIR = "/kaggle/working/output"
ADAPTER_DIR = os.path.join(OUTPUT_DIR, "bangla-smollm-135m-lora")
MERGED_DIR = os.path.join(OUTPUT_DIR, "bangla-smollm-135m-merged")
os.makedirs(ADAPTER_DIR, exist_ok=True)
os.makedirs(MERGED_DIR, exist_ok=True)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Load Dataset


In [ ]:
from datasets import load_dataset

print("Loading CupCake dataset from HuggingFace...")
hf_dataset = load_dataset("bayazidhs04/cupcake-liltii")
train_data = list(hf_dataset['train'])
val_data = list(hf_dataset['validation'])

all_data = train_data + val_data
random.shuffle(all_data)
print(f"Total examples: {len(all_data)}")

split_idx = int(0.9 * len(all_data))
train_dataset = Dataset.from_list(all_data[:split_idx])
val_dataset = Dataset.from_list(all_data[split_idx:])

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

## Train Model


In [ ]:
BASE_MODEL = "rnnandi/bangla-smollm-135m"
MAX_SEQ_LENGTH = 2048

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

print(f"Loading tokenizer: {BASE_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading model: {BASE_MODEL}")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

def tokenize_fn(example):
    messages = example['messages']
    if tokenizer.chat_template:
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    else:
        parts = [f"{m['role']}\n{m['content']}" for m in messages]
        prompt = "\n".join(parts)
    tokenized = tokenizer(prompt, truncation=True, max_length=MAX_SEQ_LENGTH, padding=False)
    tokenized['labels'] = tokenized['input_ids'][:]
    return tokenized

tokenized_train = train_dataset.map(tokenize_fn, remove_columns=train_dataset.column_names)
tokenized_val = val_dataset.map(tokenize_fn, remove_columns=val_dataset.column_names)
print(f"Tokenized train: {len(tokenized_train)}, val: {len(tokenized_val)}")

In [ ]:
training_args = TrainingArguments(
    output_dir=ADAPTER_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=50,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    seed=SEED,
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True, pad_to_multiple_of=8)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    processing_class=tokenizer,
)

print("\n=== Starting Training ===")
trainer.train()

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"\nAdapter saved to: {ADAPTER_DIR}")

import shutil
for fname in os.listdir(ADAPTER_DIR):
    src = os.path.join(ADAPTER_DIR, fname)
    dst = os.path.join(OUTPUT_DIR, fname)
    if os.path.isfile(src):
        shutil.copy2(src, dst)
print("Files copied to /kaggle/working/output/")